# 01 — Build the ward × year panel

Assembles the modelling table that the susceptibility model (notebook 03) and the
hotspot index (notebook 02) consume. One row per **(ward, year)** for 2005–2022:

* **target** — `flood_count` (basement-flood service requests that year)
* **static infrastructure features** — sewer age/length/diameter, buried-river density,
  river–sewer co-location (same for every year of a ward)
* **time-varying driver** — that year's city-wide rainfall, plus river/sewer × rainfall
  interaction terms

All the heavy lifting lives in `modelling/lib.py`; this notebook orchestrates and inspects.

> **Caveat.** The flood xlsx is on Toronto's old **44-ward** system; the only ward geometry
> we have is the current **25-ward** boundaries. We build on 25-ward geometry and join floods
> by ward number — an approximation, since ward numbers are not geographically equivalent
> between the systems. See the `lib.py` docstring.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parents[0]))   # modelling/ on the path
import lib
import pandas as pd
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 40)
print('repo :', lib.REPO)
print('raw  :', lib.RAW)
print('out  :', lib.processed_dir())

repo : /h/u7/c3/05/fuhrerhu/hidden-rivers
raw  : /h/u7/c3/05/fuhrerhu/hidden-rivers/data/raw
out  : /h/u7/c3/05/fuhrerhu/hidden-rivers/data/processed


## Load & inspect the raw inputs

In [2]:
wards   = lib.load_wards()
sewers  = lib.load_sewers()
rivers  = lib.load_rivers()
floods  = lib.load_basement_floods_long()
precip  = lib.load_precip_annual()

print(f'wards  : {len(wards):>5}  (ward numbers {sorted(wards.ward_num)[0]}–{sorted(wards.ward_num)[-1]})')
print(f'sewers : {len(sewers):>5}  install {int(sewers.install_year.min())}–{int(sewers.install_year.max())}')
print(f'rivers : {len(rivers):>5}  buried  {int(rivers.last_year.min())}–{int(rivers.last_year.max())}')
print(f'floods : {len(floods):>5}  rows ({floods.year.min()}–{floods.year.max()}, {floods.ward_num.nunique()} wards)')
print(f'precip : {len(precip):>5}  annual rows')
precip.tail(3)

wards  :    25  (ward numbers 1–25)
sewers :  3200  install 1886–2026
rivers :   458  buried  1802–2017
floods :   792  rows (2005–2022, 44 wards)
precip :    63  annual rows


,year,precip_mm,rain_mm,snow_cm,heavy_days,precip_hours
60,2020,793.9,647.5,102.55,1,1413.0
61,2021,873.8,765.8,75.74,2,1421.0
62,2022,867.7,699.7,118.02,4,1197.0


## Per-ward static infrastructure features

`weak_sewer_km` = length of sewers installed on/before 1960 (aging-capacity proxy);
`river_sewer_overlap_km` = buried river lying within 100 m of a trunk sewer;
`river_x_weaksewer` = the static hidden-river × weak-sewer coincidence term.

In [3]:
static = lib.build_ward_static_features(wards, sewers, rivers)
print('static features:', static.shape)
cols = ['ward_num','ward_name','area_km2','sewer_len_km','sewer_age','pct_old_sewer',
        'weak_sewer_km','mean_diameter','river_density','river_sewer_overlap_km','river_x_weaksewer']
static[cols].sort_values('river_x_weaksewer', ascending=False).round(2).head(10)

static features: (25, 17)


,ward_num,ward_name,area_km2,sewer_len_km,sewer_age,pct_old_sewer,weak_sewer_km,mean_diameter,river_density,river_sewer_overlap_km,river_x_weaksewer
4,19,Beaches-East York,16.80,12.41,66.90,0.97,12.04,1461.33,1.77,3.10,1.71
17,8,Eglinton-Lawrence,22.66,20.57,70.38,0.66,13.63,1656.05,2.19,19.65,1.45
21,13,Toronto Centre,5.86,13.45,78.08,0.42,5.60,2557.08,3.18,7.90,1.32
19,16,Don Valley East,22.96,13.88,63.67,0.86,11.94,1235.02,1.05,7.58,0.91
11,15,Don Valley West,30.29,29.27,75.23,0.85,24.80,1968.97,0.94,9.84,0.79
12,12,Toronto-St. Paul's,13.12,8.00,72.47,0.51,4.07,2727.68,1.46,6.21,0.74
22,5,York South-Weston,24.97,26.77,71.39,0.77,20.52,1467.64,0.96,7.01,0.74
20,9,Davenport,12.09,2.48,63.37,0.32,0.80,1852.99,1.88,4.08,0.61
23,14,Toronto-Danforth,21.80,23.59,74.84,0.47,11.17,2868.45,1.11,6.59,0.52
13,4,Parkdale-High Park,15.32,3.44,85.14,0.61,2.08,1369.17,0.83,0.31,0.50


In [4]:
static[lib.STATIC_FEATURES].describe().round(2)

,area_km2,sewer_len_km,sewer_density,sewer_age,mean_install_year,pct_old_sewer,weak_sewer_km,mean_diameter,pct_small_diam,river_len_km,river_density,river_sewer_overlap_km,river_x_weaksewer
count,25.00,25.00,25.00,25.00,25.00,25.00,25.00,25.00,25.00,25.00,25.00,25.00,25.00
mean,25.70,14.33,0.61,64.88,1957.12,0.50,7.75,1558.70,0.14,14.44,0.80,4.22,0.43
std,11.26,8.36,0.45,8.43,8.43,0.24,6.43,722.16,0.15,14.19,0.86,5.15,0.51
min,5.86,2.48,0.16,50.46,1936.86,0.00,0.02,858.14,0.00,0.00,0.00,0.00,0.00
25%,18.70,8.00,0.34,60.38,1950.61,0.32,3.37,1033.64,0.01,0.00,0.00,0.00,0.00
50%,24.42,13.45,0.50,62.41,1959.59,0.51,6.57,1235.44,0.07,15.16,0.83,2.48,0.34
75%,30.43,20.39,0.78,71.39,1961.62,0.61,11.17,1852.99,0.23,24.09,1.11,7.01,0.74
max,54.09,33.78,2.29,85.14,1971.54,0.97,24.80,3646.28,0.43,49.56,3.18,19.65,1.71


## Assemble the ward × year panel

In [5]:
panel = lib.build_panel(static)
print('panel:', panel.shape, '|', panel.ward_num.nunique(), 'wards ×', panel.year.nunique(), 'years')
print('target flood_count — mean %.1f, median %.0f, max %.0f'
      % (panel.flood_count.mean(), panel.flood_count.median(), panel.flood_count.max()))
panel[['ward_num','year','flood_count','sewer_age','pct_old_sewer','river_density',
       'heavy_days','river_x_heavy','weaksewer_x_heavy']].head(6).round(2)

  note: dropped 19 ward(s) from flood data with no 25-ward geometry (44->25 ward mismatch — see lib.py caveat)
panel: (450, 26) | 25 wards × 18 years
target flood_count — mean 244.2, median 211, max 829


,ward_num,year,flood_count,sewer_age,pct_old_sewer,river_density,heavy_days,river_x_heavy,weaksewer_x_heavy
0,1,2005,180.0,60.62,0.32,0.00,2,0.00,13.13
1,2,2005,183.0,62.29,0.54,0.00,2,0.00,18.19
2,3,2005,130.0,62.91,0.54,0.00,2,0.00,36.15
3,4,2005,166.0,85.14,0.61,0.83,2,1.65,4.16
4,5,2005,386.0,71.39,0.77,0.96,2,1.92,41.05
5,6,2005,437.0,55.65,0.44,0.96,2,1.92,18.11


In [6]:
# Quick signal check: do floods rise in heavy-rain years and in old/buried-river wards?
print('corr(flood_count, heavy_days)      = %+.2f' % panel['flood_count'].corr(panel['heavy_days']))
print('corr(flood_count, pct_old_sewer)   = %+.2f' % panel['flood_count'].corr(panel['pct_old_sewer']))
print('corr(flood_count, river_density)   = %+.2f' % panel['flood_count'].corr(panel['river_density']))
print('corr(flood_count, river_x_weaksewer)=%+.2f' % panel['flood_count'].corr(panel['river_x_weaksewer']))

corr(flood_count, heavy_days)      = +0.23
corr(flood_count, pct_old_sewer)   = +0.08
corr(flood_count, river_density)   = +0.18
corr(flood_count, river_x_weaksewer)=+0.12


## Save artifacts

`ward_year_panel.csv` → model (notebook 03) · `ward_static_features.geojson` → hotspot map (notebook 02) & the app.

In [7]:
out = lib.processed_dir()
panel.to_csv(out / 'ward_year_panel.csv', index=False)
static.to_file(out / 'ward_static_features.geojson', driver='GeoJSON')
print('wrote', out / 'ward_year_panel.csv', f'({len(panel)} rows)')
print('wrote', out / 'ward_static_features.geojson', f'({len(static)} wards)')

wrote /h/u7/c3/05/fuhrerhu/hidden-rivers/data/processed/ward_year_panel.csv (450 rows)
wrote /h/u7/c3/05/fuhrerhu/hidden-rivers/data/processed/ward_static_features.geojson (25 wards)
